In [1]:
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error

In [2]:
# Specify input and output paths
output_path = '-- precise output path, where the model will be saved --'
input_path = '-- precise input_path, where the datasets are saved --'

In [3]:
# Open training and testing sets
test = pd.read_pickle(input_path + 'test.pkl')
train = pd.read_pickle(input_path + 'train.pkl')

In [5]:
# Input column names
lsfeat = ['AL', 'K', 'AQD', 'LT', 'CCT', 'WTW', 'ES_postop']
# Target column name
target = 'Power'

In [6]:
X_train = train[lsfeat].values
X_test = test[lsfeat].values
y_train = train[target].values
y_test = test[target].values

In [7]:
# DMatrix creation
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

In [8]:
# Initial parameters
params = {
    'max_depth':6,
    'min_child_weight': 1,
    'eta':.3,
    'subsample': 1,
    'colsample_bytree': 1,
    'objective':'reg:squarederror',
}

In [9]:
params['eval_metric'] = "mae"
num_boost_round = 999

In [10]:
model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")],
    early_stopping_rounds=10
)

[0]	Test-mae:1.63047
[1]	Test-mae:1.19487
[2]	Test-mae:0.90352
[3]	Test-mae:0.69281
[4]	Test-mae:0.55119
[5]	Test-mae:0.45502
[6]	Test-mae:0.39398
[7]	Test-mae:0.35761
[8]	Test-mae:0.33112
[9]	Test-mae:0.31618
[10]	Test-mae:0.30256
[11]	Test-mae:0.29691
[12]	Test-mae:0.29168
[13]	Test-mae:0.28814
[14]	Test-mae:0.28364
[15]	Test-mae:0.28155
[16]	Test-mae:0.27925
[17]	Test-mae:0.27798
[18]	Test-mae:0.27572
[19]	Test-mae:0.27495
[20]	Test-mae:0.27367
[21]	Test-mae:0.27367
[22]	Test-mae:0.27335
[23]	Test-mae:0.27335
[24]	Test-mae:0.27184
[25]	Test-mae:0.27120
[26]	Test-mae:0.26964
[27]	Test-mae:0.26904
[28]	Test-mae:0.26911
[29]	Test-mae:0.26930
[30]	Test-mae:0.26879
[31]	Test-mae:0.26822
[32]	Test-mae:0.26738
[33]	Test-mae:0.26675
[34]	Test-mae:0.26633
[35]	Test-mae:0.26603
[36]	Test-mae:0.26537
[37]	Test-mae:0.26569
[38]	Test-mae:0.26547
[39]	Test-mae:0.26478
[40]	Test-mae:0.26410
[41]	Test-mae:0.26515
[42]	Test-mae:0.26430
[43]	Test-mae:0.26483
[44]	Test-mae:0.26444
[45]	Test-mae:0.2645

In [11]:
cv_results = xgb.cv(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    seed=99,
    nfold=5,
    metrics={'mae'},
    early_stopping_rounds=10
)

In [12]:
# Initial MAE
cv_results['test-mae-mean'].min()

0.27378614390054684

In [13]:
# max_depth and min_child_weight tuning
gridsearch_params = [
    (max_depth, min_child_weight)
    for max_depth in range(9,12)
    for min_child_weight in range(5,8)
]

In [14]:
min_mae = float("Inf")
best_params = None
for max_depth, min_child_weight in gridsearch_params:
    print("max_depth={}, min_child_weight={}".format(
                             max_depth,
                             min_child_weight))
    # Update parameters
    params['max_depth'] = max_depth
    params['min_child_weight'] = min_child_weight
    # Run CV
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        seed=42,
        nfold=5,
        metrics={'mae'},
        early_stopping_rounds=10
    )
    # Update best MAE
    mean_mae = cv_results['test-mae-mean'].min()
    boost_rounds = cv_results['test-mae-mean'].argmin()
    print("\tMAE {} after {} rounds".format(mean_mae, boost_rounds))
    if mean_mae < min_mae:
        min_mae = mean_mae
        best_params = (max_depth,min_child_weight)
print("Best params: {}, {}, MAE: {}".format(best_params[0], best_params[1], min_mae))

max_depth=9, min_child_weight=5
	MAE 0.27036596921992007 after 58 rounds
max_depth=9, min_child_weight=6
	MAE 0.2688015776547226 after 52 rounds
max_depth=9, min_child_weight=7
	MAE 0.26925288858422597 after 70 rounds
max_depth=10, min_child_weight=5
	MAE 0.2728362280773316 after 47 rounds
max_depth=10, min_child_weight=6
	MAE 0.2741516472887996 after 36 rounds
max_depth=10, min_child_weight=7
	MAE 0.271751425052563 after 51 rounds
max_depth=11, min_child_weight=5
	MAE 0.27407208499167124 after 69 rounds
max_depth=11, min_child_weight=6
	MAE 0.27311096692279957 after 42 rounds
max_depth=11, min_child_weight=7
	MAE 0.2740335182299351 after 44 rounds
Best params: 9, 6, MAE: 0.2688015776547226


In [15]:
# best max_depth and min_child_weight are specified
params['max_depth'] = best_params[0]
params['min_child_weight'] = best_params[1]

In [16]:
# subsample and colsample tuning
gridsearch_params = [
    (subsample, colsample)
    for subsample in [i/10. for i in range(7,11)]
    for colsample in [i/10. for i in range(7,11)]
]

In [17]:
min_mae = float("Inf")
best_params = None

for subsample, colsample in reversed(gridsearch_params):
    print("subsample={}, colsample={}".format(
                             subsample,
                             colsample))
    # Update parameters
    params['subsample'] = subsample
    params['colsample_bytree'] = colsample
    # Run CV
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        seed=42,
        nfold=5,
        metrics={'mae'},
        early_stopping_rounds=10
    )
    # Update best score
    mean_mae = cv_results['test-mae-mean'].min()
    boost_rounds = cv_results['test-mae-mean'].argmin()
    print("\tMAE {} after {} rounds".format(mean_mae, boost_rounds))
    if mean_mae < min_mae:
        min_mae = mean_mae
        best_params = (subsample,colsample)
        
print("Best params: {}, {}, MAE: {}".format(best_params[0], best_params[1], min_mae))

subsample=1.0, colsample=1.0
	MAE 0.2688015776547226 after 52 rounds
subsample=1.0, colsample=0.9
	MAE 0.41531695885705544 after 79 rounds
subsample=1.0, colsample=0.8
	MAE 0.4419064959001586 after 116 rounds
subsample=1.0, colsample=0.7
	MAE 0.483221400514038 after 213 rounds
subsample=0.9, colsample=1.0
	MAE 0.2754096166306591 after 50 rounds
subsample=0.9, colsample=0.9
	MAE 0.41862636713102586 after 65 rounds
subsample=0.9, colsample=0.8
	MAE 0.437428215443569 after 141 rounds
subsample=0.9, colsample=0.7
	MAE 0.47544382006849073 after 256 rounds
subsample=0.8, colsample=1.0
	MAE 0.27884257137957397 after 33 rounds
subsample=0.8, colsample=0.9
	MAE 0.42843614060792373 after 86 rounds
subsample=0.8, colsample=0.8
	MAE 0.44704425601652586 after 66 rounds
subsample=0.8, colsample=0.7
	MAE 0.48008893916823075 after 110 rounds
subsample=0.7, colsample=1.0
	MAE 0.27773704263712135 after 48 rounds
subsample=0.7, colsample=0.9
	MAE 0.42260312314468 after 106 rounds
subsample=0.7, colsample

In [18]:
# best subsample and colsample values are specified
params['subsample'] = best_params[0]
params['colsample_bytree'] = best_params[1]

In [19]:
# eta tuning
min_mae = float("Inf")
best_params = None
for eta in [.3, .2, .1, .05, .01, .005]:
    print("eta={}".format(eta))
    # Update parameters
    params['eta'] = eta
    # Run and time CV
    cv_results = xgb.cv(
            params,
            dtrain,
            num_boost_round=num_boost_round,
            seed=42,
            nfold=5,
            metrics=['mae'],
            early_stopping_rounds=10
          )
    # Update best score
    mean_mae = cv_results['test-mae-mean'].min()
    boost_rounds = cv_results['test-mae-mean'].argmin()
    print("\tMAE {} after {} rounds\n".format(mean_mae, boost_rounds))
    if mean_mae < min_mae:
        min_mae = mean_mae
        best_params = eta
print("Best params: {}, MAE: {}".format(best_params, min_mae))

eta=0.3
	MAE 0.2688015776547226 after 52 rounds

eta=0.2
	MAE 0.26151587683840993 after 84 rounds

eta=0.1
	MAE 0.25747269753040375 after 162 rounds

eta=0.05
	MAE 0.25364017138276307 after 326 rounds

eta=0.01
	MAE 0.25422000732907846 after 997 rounds

eta=0.005
	MAE 0.2629005760268476 after 998 rounds

Best params: 0.05, MAE: 0.25364017138276307


In [20]:
# best eta is specified
params['eta'] = best_params

In [21]:
model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")],
    early_stopping_rounds=10
)

print("Best MAE: {:.2f} in {} rounds".format(model.best_score, model.best_iteration+1))

[0]	Test-mae:2.13230
[1]	Test-mae:2.03081
[2]	Test-mae:1.93425
[3]	Test-mae:1.84332
[4]	Test-mae:1.75577
[5]	Test-mae:1.67304
[6]	Test-mae:1.59492
[7]	Test-mae:1.52149
[8]	Test-mae:1.45197
[9]	Test-mae:1.38511
[10]	Test-mae:1.32322
[11]	Test-mae:1.26368
[12]	Test-mae:1.20733
[13]	Test-mae:1.15281
[14]	Test-mae:1.10257
[15]	Test-mae:1.05342
[16]	Test-mae:1.00884
[17]	Test-mae:0.96569
[18]	Test-mae:0.92499
[19]	Test-mae:0.88598
[20]	Test-mae:0.84863
[21]	Test-mae:0.81322
[22]	Test-mae:0.77971
[23]	Test-mae:0.74767
[24]	Test-mae:0.71740
[25]	Test-mae:0.69001
[26]	Test-mae:0.66269
[27]	Test-mae:0.63691
[28]	Test-mae:0.61307
[29]	Test-mae:0.59108
[30]	Test-mae:0.56996
[31]	Test-mae:0.54981
[32]	Test-mae:0.53122
[33]	Test-mae:0.51337
[34]	Test-mae:0.49711
[35]	Test-mae:0.48107
[36]	Test-mae:0.46665
[37]	Test-mae:0.45351
[38]	Test-mae:0.44028
[39]	Test-mae:0.42819
[40]	Test-mae:0.41761
[41]	Test-mae:0.40694
[42]	Test-mae:0.39770
[43]	Test-mae:0.38828
[44]	Test-mae:0.37973
[45]	Test-mae:0.3713

In [22]:
num_boost_round = model.best_iteration + 1
best_model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")]
)

[0]	Test-mae:2.13230
[1]	Test-mae:2.03081
[2]	Test-mae:1.93425
[3]	Test-mae:1.84332
[4]	Test-mae:1.75577
[5]	Test-mae:1.67304
[6]	Test-mae:1.59492
[7]	Test-mae:1.52149
[8]	Test-mae:1.45197
[9]	Test-mae:1.38511
[10]	Test-mae:1.32322
[11]	Test-mae:1.26368
[12]	Test-mae:1.20733
[13]	Test-mae:1.15281
[14]	Test-mae:1.10257
[15]	Test-mae:1.05342
[16]	Test-mae:1.00884
[17]	Test-mae:0.96569
[18]	Test-mae:0.92499
[19]	Test-mae:0.88598
[20]	Test-mae:0.84863
[21]	Test-mae:0.81322
[22]	Test-mae:0.77971
[23]	Test-mae:0.74767
[24]	Test-mae:0.71740
[25]	Test-mae:0.69001
[26]	Test-mae:0.66269
[27]	Test-mae:0.63691
[28]	Test-mae:0.61307
[29]	Test-mae:0.59108
[30]	Test-mae:0.56996
[31]	Test-mae:0.54981
[32]	Test-mae:0.53122
[33]	Test-mae:0.51337
[34]	Test-mae:0.49711
[35]	Test-mae:0.48107
[36]	Test-mae:0.46665
[37]	Test-mae:0.45351
[38]	Test-mae:0.44028
[39]	Test-mae:0.42819
[40]	Test-mae:0.41761
[41]	Test-mae:0.40694
[42]	Test-mae:0.39770
[43]	Test-mae:0.38828
[44]	Test-mae:0.37973
[45]	Test-mae:0.3713

In [23]:
mean_absolute_error(best_model.predict(dtest), y_test)

0.2527917463388016

In [24]:
best_model.save_model(output_path + "AItoPower.model")